# Experiment 3 — Wheat Flour Protein (Spectrometer → Hyperspectral Camera)

**Paper:** *Transferability Loss for Safe Model Selection under Domain Shift* (ICLR 2026)

**Dataset:** Wheat flour protein content measured with two NIR instruments — a benchtop spectrometer (source) and a hyperspectral camera (target). The instrument change induces a natural covariate shift between the two domains.

**Objective:** Evaluate TR-LOSS as a model-selection criterion for a real-world instrument-transfer regression task, where target-domain labels are unavailable at selection time.

**Protocol:**
1. Fit KDA-PLS regression models (linear kernel) over a grid of latent components (`n_comp`) and regularization strength (`λ`) for the single spectrometer → camera transfer pair.
2. Compute the following quantities per model:
   - **RMSEV\_S** — source validation RMSE
   - **τ\_rMSE** (TR\_LOSS) — transferability loss between paired target-domain predictions
   - **RMSEV\_T** — target validation RMSE (oracle only)
   - **RMSEP\_T** — target test RMSE (final evaluation)
   - **MMD** — maximum mean discrepancy between source and target latent scores
3. Select models by minimising the composite criterion $J_{\mathrm{reg}} = \mathrm{MSE}_S + \tau_{\mathrm{MSE}}$ (Eq. 10) and compare against source-only, MMD-based, and oracle baselines.


## Load modules 

In [54]:

import sys
sys.path.append("../../../methods/")

from kdaPLS.kdapls import KDAPLSRegression
from trloss import transferability_loss, selection_criterion
from metrics.mmd import mmd_rbf, mmd_linear

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mtply
import plotly.express as px
import plotly.graph_objects as go
import scipy.io as sp_io
import scipy as sp

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


## Load data

In [55]:
# get data (slow)
data_url = "../../../data/hypercamera"
df_s = pd.read_csv(data_url + "/spectrometer.csv", index_col = 0)
df_t = pd.read_csv(data_url + "/hypercamera.csv", index_col = 0)

Xs = np.asarray(df_s.iloc[:,:-1])
Xt = np.asarray(df_t.iloc[:,:-1])
ys = np.asarray(df_s.iloc[:,-1]).reshape(-1,1)
yt = np.asarray(df_t.iloc[:,-1]).reshape(-1,1)

y_name = "protein"
data_dict = {}

data_dict["X_spectrometer"] = Xs.copy()
data_dict["Y_spectrometer"] = ys.copy()

data_dict["X_hypercamera"] = Xt.copy()
data_dict["Y_hypercamera"] = yt.copy()

domains = ["spectrometer","hypercamera"]

## Generate adaptations

In [56]:
# Hyperparameters
n_comp_list = [i for i in range(1,21)]  # Number of LVs
lambda_list = np.logspace(0.5,1,10)     # Regularization parameter
kdict = {"type":"linear"}               # Kernel

In [57]:
model_results = {}
k = 0

for source_name in domains[:1]:
    domains_source = [re for re in domains if re!=source_name]
    for target_name in domains_source:
        print(source_name, target_name)

        Xs_all = data_dict[f"X_{source_name}"]
        Xt_all = data_dict[f"X_{target_name}"]
        ys_all = data_dict[f"Y_{source_name}"]
        yt_all = data_dict[f"Y_{target_name}"]

        # Splitting

        Xs, Xs_val_test, ys, ys_val_test = train_test_split(Xs_all, ys_all, test_size=0.8, random_state=42245368)
        Xs_val, Xs_test, ys_val, ys_test = train_test_split(Xs_val_test, ys_val_test, test_size=0.8, random_state=42245368)
        Xt, Xt_val_test, yt, yt_val_test = train_test_split(Xt_all, yt_all, test_size=0.3, random_state=42245368)
        Xt_val, Xt_test, yt_val, yt_test = train_test_split(Xt_val_test, yt_val_test, test_size=0.3, random_state=42245368)



        for n_comp in n_comp_list:
            
            for lambda_ in lambda_list:

                kdapls_model = KDAPLSRegression(xs = Xs, xt = Xt, n_components = n_comp, kdict = kdict, l=[lambda_], target_domain = 0)
                kdapls_model.fit(Xs,ys)
                # Validation and test prediction
                yt_val_pred_all = kdapls_model.predict_all(Xt_val)
                ys_val_pred = kdapls_model.predict(Xs_val)
                yt_test_pred = kdapls_model.predict(Xt_test)
                # Other distance metrics                
                ns = Xs.shape[0]
                nt = Xt.shape[0]
                mmd = mmd_linear(kdapls_model.x_scores_st_[:ns,:],kdapls_model.x_scores_st_[ns:,:])
                # Performance metrics
                tr_loss = np.sqrt(transferability_loss(yt_val_pred_all[0], yt_val_pred_all[1]))
                rmsev = np.sqrt(mean_squared_error(ys_val, ys_val_pred))
                r2v = r2_score(ys_val, ys_val_pred)
                rmsev_t = np.sqrt(mean_squared_error(yt_val, yt_val_pred_all[0]))
                r2v_t = r2_score(yt_val, yt_val_pred_all[0])
                rmsep_t = np.sqrt(mean_squared_error(yt_test, yt_test_pred))
                r2p_t = r2_score(yt_test, yt_test_pred)
                
                model_results[k] = {"TR_LOSS":tr_loss,
                                     "RMSEV_S":rmsev,
                                     "r2v":r2v,
                                    "RMSEV_T":rmsev_t,
                                      "r2v_t":r2v_t,
                                      "RMSEP_T":rmsep_t,
                                      "r2p_t":r2p_t,
                                      "n_comp":n_comp,
                                      "lambda_":lambda_ ,    
                                    "Source":source_name,
                                    "Target":target_name,
                                    "MMD": mmd
                                    
                                    

                                                         }
                k += 1
                
print("done")

spectrometer hypercamera
done


## Model selection

In [58]:
model_results_pd = pd.DataFrame.from_dict(model_results, orient="index")
model_results_pd = model_results_pd.loc[model_results_pd["Source"]=="spectrometer"].copy()
model_results_pd["rMSEV_S"] = model_results_pd["RMSEV_S"]
model_results_pd["rTR_LOSS"] = model_results_pd["TR_LOSS"]
model_results_pd["rMSEV_T"] = model_results_pd["RMSEV_T"]
model_results_pd["rMSEP_T"] = model_results_pd["RMSEP_T"]
model_results_pd["domains"] = model_results_pd.apply(lambda x: x["Source"]+"->"+x["Target"], axis = 1)
model_results_pd["model_id"] = np.arange(1, model_results_pd.shape[0]+1)
model_metrics_pd_melt = pd.melt(model_results_pd.loc[:, ["model_id","domains","rMSEV_S","MMD","rTR_LOSS","rMSEV_T","rMSEP_T"]], id_vars = ["model_id","domains","rMSEV_S","rMSEP_T"], var_name = "measure_alignment")
model_metrics_pd_melt['measure_alignment'] = model_metrics_pd_melt['measure_alignment'].replace('rTR_LOSS', 'τ<sub>rMSE</sub>')

# Compute selection criterion on filtered data and find selected/oracle models
mr = model_results_pd.dropna().copy()
# Eq. 10: J_reg = MSE_S + τ_MSE  →  minimize
mr["RMSEV_S + TR_LOSS"] = selection_criterion(
    mr["RMSEV_S"]**2, mr["TR_LOSS"]**2, task="regression"
)

best_trloss = mr.loc[mr["RMSEV_S + TR_LOSS"].idxmin()]
best_oracle = mr.loc[mr["RMSEP_T"].idxmin()]

fig = px.scatter(model_metrics_pd_melt, x = "value", y = "rMSEV_S", color = "rMSEP_T", 
                 hover_data=["model_id"],
                 facet_col = "measure_alignment", 
                 category_orders={"measure_alignment": ["MMD", "τ<sub>rMSE</sub>","rMSEV_T"]},
                 color_continuous_scale=px.colors.sequential.Viridis_r,
                 range_color=[1,3.5],
                )
fig.update_xaxes(matches=None)
fig.update_traces(marker=dict(size=10, opacity=0.4))
fig.update_layout(width = 1000, height = 300, font_size = 18,
                  legend=dict(
        yanchor="top", y=0.85,
        xanchor="left", x=0.425,
        bgcolor="rgba(255,255,255,0.8)",
        font=dict(size=11)))

# Overlay selected (red cross) and oracle (blue square) on τ_rMSE panel only (col 2)
fig.add_trace(
    go.Scatter(x=[best_trloss["TR_LOSS"]], y=[best_trloss["RMSEV_S"]],
               mode="markers",
               marker=dict(symbol="cross", size=14, color="red", opacity=1.0, line=dict(width=2, color="darkred")),
               name="Selected (τ)", legendgroup="selected", showlegend=True,
               hovertemplate=(f"n_comp={int(best_trloss['n_comp'])}, λ={best_trloss['lambda_']:.2g}<br>"
                              f"RMSEP_T={best_trloss['RMSEP_T']:.3f}<extra></extra>")),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[best_oracle["TR_LOSS"]], y=[best_oracle["RMSEV_S"]],
               mode="markers",
               marker=dict(symbol="square", size=12, color="blue", opacity=1.0, line=dict(width=1.5, color="black")),
               name="Oracle (best RMSEP<sub>T</sub>)", legendgroup="oracle", showlegend=True,
               hovertemplate=(f"n_comp={int(best_oracle['n_comp'])}, λ={best_oracle['lambda_']:.2g}<br>"
                              f"RMSEP_T={best_oracle['RMSEP_T']:.3f}<extra></extra>")),
    row=1, col=2
)

# Update facet titles: replace "measure_alignment=X" with "value: X"
for annotation in fig.layout.annotations:
    if "measure_alignment=" in annotation.text:
        label = annotation.text.replace("measure_alignment=", "")
        annotation.text = f"value: {label}"

# fig.write_image("../output/03_hypercamera.pdf")
fig.show()


# Final models

In [59]:
# TR-LOSS criterion (Eq. 10 from paper): J_reg = MSE_S + τ_MSE  →  minimize
model_results_pd["RMSEV_S + TR_LOSS"] = selection_criterion(
    model_results_pd["RMSEV_S"]**2, model_results_pd["TR_LOSS"]**2, task="regression"
)

# MMD criterion: normalized Euclidean (MMD is in different units)
modelsel_criterion_mmd = np.sqrt((model_results_pd["MMD"]/np.amax(model_results_pd["MMD"]))**2 + (model_results_pd["RMSEV_S"]/np.amax(model_results_pd["RMSEV_S"]))**2)
model_results_pd["RMSEV_S + MMD"] = modelsel_criterion_mmd


final_models_dict = {}
kk = 0
     
model_result_st = model_results_pd.copy()
final_models_dict[kk] = {
                        "Source":source_name,
                        "Target":target_name,
                        "RMSEV_S": model_result_st.sort_values("RMSEV_S").iloc[0]["RMSEP_T"],   
                        "RMSEV_S + MMD": model_result_st.sort_values("RMSEV_S + MMD").iloc[0]["RMSEP_T"],           
                        "RMSEV_S + TR_LOSS":model_result_st.sort_values("RMSEV_S + TR_LOSS").iloc[0]["RMSEP_T"],
                        "RMSEV_T": model_result_st.sort_values("RMSEV_T").iloc[0]["RMSEP_T"],}

kk += 1
        
final_models_pd = pd.DataFrame.from_dict(final_models_dict, orient="index")
# final_models_pd.to_csv("../output/01_melamine_dapls.csv")
print(final_models_pd.to_latex(index=False,float_format="{:.2f}".format))


\begin{tabular}{llrrrr}
\toprule
Source & Target & RMSEV_S & RMSEV_S + MMD & RMSEV_S + TR_LOSS & RMSEV_T \\
\midrule
spectrometer & hypercamera & 2.01 & 2.06 & 1.24 & 1.02 \\
\bottomrule
\end{tabular}

